In [ ]:
# Mortgage Complaints - Data Exploration & Cleaning
# This notebook loads, explores, cleans, and prepares mortgage complaint data for BigQuery

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 100)

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Load raw mortgage data
print("Loading mortgage complaint data...")
mortgage_data = pd.read_csv("raw/mortgage.csv")

print(f"✓ Data loaded successfully!")
print(f"Shape: {mortgage_data.shape[0]:,} rows × {mortgage_data.shape[1]} columns")
print(f"Memory usage: {mortgage_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nFirst 5 rows:")
mortgage_data.head()

In [ ]:
# Geographic and Product Analysis
print("GEOGRAPHIC & PRODUCT DISTRIBUTION")
print("=" * 80)

state_counts = mortgage_data['State'].value_counts()
print(f"\nTotal unique states: {len(state_counts)}")
print(f"\nTop 20 States by Complaint Volume:")
print(state_counts.head(20))

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Top 20 states
ax = axes[0, 0]
top_states = state_counts.head(20)
ax.barh(range(len(top_states)), top_states.values, color='skyblue')
ax.set_yticks(range(len(top_states)))
ax.set_yticklabels(top_states.index, fontsize=9)
ax.set_xlabel('Number of Complaints', fontsize=10)
ax.set_title('Top 20 States by Complaint Volume', fontsize=12, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(top_states.values):
    ax.text(v + 50, i, f'{v:,}', va='center', fontsize=8)

# 2. Sub-product distribution (top 10)
ax = axes[0, 1]
subproduct_counts = mortgage_data['Sub-product'].value_counts().head(10)
ax.barh(range(len(subproduct_counts)), subproduct_counts.values, color='lightgreen')
ax.set_yticks(range(len(subproduct_counts)))
ax.set_yticklabels([p[:35] for p in subproduct_counts.index], fontsize=8)
ax.set_xlabel('Number of Complaints', fontsize=10)
ax.set_title('Top 10 Mortgage Sub-Products', fontsize=12, fontweight='bold')
ax.invert_yaxis()

# 3. ZIP code data quality
ax = axes[1, 0]
zip_quality = {
    'Complete ZIP': mortgage_data['ZIP code'].notna().sum(),
    'Missing ZIP': mortgage_data['ZIP code'].isna().sum(),
    'Masked ZIP (XXX)': mortgage_data['ZIP code'].astype(str).str.contains('XX', na=False).sum()
}
ax.bar(zip_quality.keys(), zip_quality.values(), color=['green', 'red', 'orange'])
ax.set_ylabel('Count', fontsize=10)
ax.set_title('ZIP Code Data Quality', fontsize=12, fontweight='bold')
ax.set_xticklabels(zip_quality.keys(), rotation=15, ha='right')
for i, (k, v) in enumerate(zip_quality.items()):
    ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

# 4. Company response types
ax = axes[1, 1]
response_counts = mortgage_data['Company response to consumer'].value_counts().head(8)
ax.bar(range(len(response_counts)), response_counts.values, color=sns.color_palette('pastel'))
ax.set_xticks(range(len(response_counts)))
ax.set_xticklabels([r[:20] for r in response_counts.index], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Count', fontsize=10)
ax.set_title('Company Response Types', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Export cleaned data
import os

print("EXPORTING CLEANED DATA")
print("=" * 80)

# Create output directory if it doesn't exist
output_dir = '../../../data'
os.makedirs(output_dir, exist_ok=True)

# Export to CSV
output_path = os.path.join(output_dir, 'mortgage_cleaned.csv')
mortgage_cleaned.to_csv(output_path, index=False)

print(f"✓ Data exported successfully!")
print(f"  Output path: {output_path}")
print(f"  File size: {os.path.getsize(output_path) / 1024**2:.2f} MB")
print(f"  Total records: {len(mortgage_cleaned):,}")

print("\n" + "=" * 80)
print("MORTGAGE DATA CLEANING COMPLETE!")
print("=" * 80)
print(f"\nNext steps:")
print(f"1. Review the cleaned data at: {output_path}")
print(f"2. Optionally combine with credit card data for unified dataset")
print(f"3. Load to BigQuery using: ingestion/load_bigquery.py")

In [ ]:
# Final Data Quality Summary
print("\n" + "=" * 80)
print("CLEANED DATA SUMMARY")
print("=" * 80)

print(f"\nDataset Shape: {mortgage_cleaned.shape}")
print(f"Date Range: {mortgage_cleaned['date_received'].min().date()} to {mortgage_cleaned['date_received'].max().date()}")
print(f"Unique Companies: {mortgage_cleaned['company'].nunique():,}")
print(f"Unique Issues: {mortgage_cleaned['issue'].nunique()}")
print(f"Records with Narratives: {mortgage_cleaned['consumer_complaint_narrative'].notna().sum():,} ({mortgage_cleaned['consumer_complaint_narrative'].notna().sum()/len(mortgage_cleaned)*100:.1f}%)")

print("\n" + "-" * 80)
print("Missing Values in Cleaned Data:")
print("-" * 80)
missing_cleaned = mortgage_cleaned.isnull().sum()
missing_pct_cleaned = (missing_cleaned / len(mortgage_cleaned)) * 100
missing_summary = pd.DataFrame({
    'Column': missing_cleaned.index,
    'Missing': missing_cleaned.values,
    'Percentage': missing_pct_cleaned.values
}).sort_values('Missing', ascending=False)
print(missing_summary[missing_summary['Missing'] > 0].to_string(index=False))

print("\n" + "-" * 80)
print("Sample of Cleaned Data:")
print("-" * 80)
mortgage_cleaned.head()

In [ ]:
# DATA CLEANING - Step 7: Select and order final columns for BigQuery
print("\n6. Selecting final columns for BigQuery...")

# Define final schema for BigQuery
final_columns = [
    'complaint_id',
    'date_received',
    'date_sent_to_company',
    'product',
    'sub_product',
    'issue',
    'sub_issue',
    'consumer_complaint_narrative',
    'company',
    'state',
    'zip_code',
    'submitted_via',
    'company_response',
    'timely_response',
    'company_public_response',
    'tags'
]

mortgage_cleaned = mortgage_cleaned[final_columns]

print(f"✓ Final schema defined")
print(f"  Columns: {len(final_columns)}")
print(f"  Rows: {len(mortgage_cleaned):,}")

In [ ]:
# DATA CLEANING - Step 6: Remove duplicates
print("\n5. Removing duplicate records...")

# Remove duplicates based on complaint_id (should be unique)
before_dedup = len(mortgage_cleaned)
mortgage_cleaned = mortgage_cleaned.drop_duplicates(subset=['complaint_id'], keep='first')
after_dedup = len(mortgage_cleaned)
removed = before_dedup - after_dedup

print(f"✓ Duplicates removed")
print(f"  Records removed: {removed:,}")
print(f"  Final record count: {after_dedup:,}")

In [ ]:
# DATA CLEANING - Step 5: Handle ZIP codes
print("\n4. Processing ZIP codes...")

def clean_zip_code(zip_code):
    """Clean ZIP code - keep first 5 digits only."""
    if pd.isna(zip_code):
        return None
    zip_str = str(zip_code).strip()
    # Remove XX masks
    if 'XX' in zip_str or 'xx' in zip_str:
        return None
    # Keep only first 5 digits
    digits = ''.join(filter(str.isdigit, zip_str))
    return digits[:5] if len(digits) >= 5 else None

mortgage_cleaned['zip_code'] = mortgage_cleaned['zip_code'].apply(clean_zip_code)

valid_zips = mortgage_cleaned['zip_code'].notna().sum()
print(f"✓ ZIP codes cleaned")
print(f"  Valid ZIP codes: {valid_zips:,} ({valid_zips/len(mortgage_cleaned)*100:.1f}%)")

In [ ]:
# DATA CLEANING - Step 4: Clean text fields
print("\n3. Cleaning text fields...")

def clean_text(text):
    """Clean text by stripping whitespace and converting empty strings to None."""
    if pd.isna(text):
        return None
    text = str(text).strip()
    return text if text and text != '' else None

# Text columns to clean
text_columns = [
    'product', 'sub_product', 'issue', 'sub_issue',
    'consumer_complaint_narrative', 'company_public_response',
    'company', 'state', 'submitted_via', 'company_response',
    'timely_response', 'tags'
]

for col in text_columns:
    if col in mortgage_cleaned.columns:
        mortgage_cleaned[col] = mortgage_cleaned[col].apply(clean_text)

print(f"✓ Text fields cleaned")
print(f"  Cleaned {len(text_columns)} text columns")

In [ ]:
# DATA CLEANING - Step 3: Convert date columns
print("\n2. Converting date columns to datetime format...")

# Parse dates
mortgage_cleaned['date_received'] = pd.to_datetime(mortgage_cleaned['date_received'], errors='coerce')
mortgage_cleaned['date_sent_to_company'] = pd.to_datetime(mortgage_cleaned['date_sent_to_company'], errors='coerce')

# Check for parsing errors
date_errors = mortgage_cleaned['date_received'].isna().sum()
print(f"✓ Date columns converted")
print(f"  Date parsing errors: {date_errors}")
print(f"  Date range: {mortgage_cleaned['date_received'].min().date()} to {mortgage_cleaned['date_received'].max().date()}")

In [ ]:
# DATA CLEANING - Step 2: Standardize column names to snake_case
print("1. Standardizing column names to snake_case...")

# Create column name mapping
column_mapping = {
    'Date received': 'date_received',
    'Product': 'product',
    'Sub-product': 'sub_product',
    'Issue': 'issue',
    'Sub-issue': 'sub_issue',
    'Consumer complaint narrative': 'consumer_complaint_narrative',
    'Company public response': 'company_public_response',
    'Company': 'company',
    'State': 'state',
    'ZIP code': 'zip_code',
    'Tags': 'tags',
    'Submitted via': 'submitted_via',
    'Date sent to company': 'date_sent_to_company',
    'Company response to consumer': 'company_response',
    'Timely response?': 'timely_response',
    'Complaint ID': 'complaint_id'
}

mortgage_cleaned = mortgage_cleaned.rename(columns=column_mapping)
print(f"✓ Column names standardized")
print(f"  New columns: {list(mortgage_cleaned.columns)}")

In [ ]:
# DATA CLEANING - Step 1: Create cleaned copy
print("STARTING DATA CLEANING PROCESS")
print("=" * 80)

# Create a copy for cleaning
mortgage_cleaned = mortgage_data.copy()
initial_rows = len(mortgage_cleaned)

print(f"Initial dataset: {initial_rows:,} rows")
print("\nCleaning steps:")
print("-" * 80)

In [ ]:
# Time Series Analysis
print("TEMPORAL ANALYSIS")
print("=" * 80)

# Convert date columns
mortgage_data['date_received_parsed'] = pd.to_datetime(mortgage_data['Date received'])
mortgage_data['date_sent_parsed'] = pd.to_datetime(mortgage_data['Date sent to company'])

# Extract time components
mortgage_data['year'] = mortgage_data['date_received_parsed'].dt.year
mortgage_data['month'] = mortgage_data['date_received_parsed'].dt.month
mortgage_data['year_month'] = mortgage_data['date_received_parsed'].dt.to_period('M')

# Monthly complaint trend
monthly_counts = mortgage_data.groupby('year_month').size()

print(f"Date range: {mortgage_data['date_received_parsed'].min()} to {mortgage_data['date_received_parsed'].max()}")
print(f"Total months covered: {len(monthly_counts)}")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Monthly trend
ax = axes[0, 0]
monthly_counts.plot(ax=ax, color='steelblue', linewidth=2)
ax.set_xlabel('Month', fontsize=10)
ax.set_ylabel('Number of Complaints', fontsize=10)
ax.set_title('Monthly Complaint Volume Trend', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# 2. Day of week distribution
mortgage_data['day_of_week'] = mortgage_data['date_received_parsed'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = mortgage_data['day_of_week'].value_counts().reindex(day_order)
ax = axes[0, 1]
ax.bar(range(len(day_counts)), day_counts.values, color='lightcoral')
ax.set_xticks(range(len(day_counts)))
ax.set_xticklabels(day_counts.index, rotation=45, ha='right')
ax.set_ylabel('Number of Complaints', fontsize=10)
ax.set_title('Complaints by Day of Week', fontsize=12, fontweight='bold')

# 3. Response time analysis
mortgage_data['response_time_days'] = (mortgage_data['date_sent_parsed'] - mortgage_data['date_received_parsed']).dt.days
ax = axes[1, 0]
response_times = mortgage_data['response_time_days'].dropna()
ax.hist(response_times[response_times <= 30], bins=30, color='lightgreen', edgecolor='black')
ax.set_xlabel('Response Time (Days)', fontsize=10)
ax.set_ylabel('Frequency', fontsize=10)
ax.set_title('Company Response Time Distribution (≤30 days)', fontsize=12, fontweight='bold')
ax.axvline(response_times.median(), color='red', linestyle='--', linewidth=2, label=f'Median: {response_times.median():.1f} days')
ax.legend()

# 4. Top companies by month (heatmap for top 5)
top_5_companies = company_counts.head(5).index
company_month_pivot = mortgage_data[mortgage_data['Company'].isin(top_5_companies)].groupby(['year_month', 'Company']).size().unstack(fill_value=0)
ax = axes[1, 1]
sns.heatmap(company_month_pivot.T, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Complaints'}, fmt='d')
ax.set_xlabel('Month', fontsize=10)
ax.set_ylabel('Company', fontsize=10)
ax.set_title('Monthly Complaints - Top 5 Companies', fontsize=12, fontweight='bold')
ax.set_yticklabels([c[:25] for c in company_month_pivot.columns], rotation=0, fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nResponse Time Statistics:")
print(f"  Mean: {response_times.mean():.2f} days")
print(f"  Median: {response_times.median():.2f} days")
print(f"  Min: {response_times.min():.2f} days")
print(f"  Max: {response_times.max():.2f} days")

In [ ]:
# Exploratory Data Analysis - Top Companies & Submission Info
print("TOP COMPANIES BY COMPLAINT VOLUME")
print("=" * 80)

company_counts = mortgage_data['Company'].value_counts()
print(f"\nTotal unique companies: {len(company_counts)}")
print(f"\nTop 20 Companies:")
print(company_counts.head(20))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Top 15 companies bar chart
ax = axes[0, 0]
top_companies = company_counts.head(15)
ax.barh(range(len(top_companies)), top_companies.values, color='coral')
ax.set_yticks(range(len(top_companies)))
ax.set_yticklabels([c[:35] for c in top_companies.index], fontsize=9)
ax.set_xlabel('Number of Complaints', fontsize=10)
ax.set_title('Top 15 Companies by Complaint Volume', fontsize=12, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(top_companies.values):
    ax.text(v + 100, i, f'{v:,}', va='center', fontsize=8)

# 2. Consumer narrative availability
ax = axes[0, 1]
narrative_counts = mortgage_data['Consumer complaint narrative'].notna().value_counts()
labels = ['No Narrative', 'Has Narrative']
colors_narrative = ['#ff9999', '#66b3ff']
ax.pie(narrative_counts.values, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors_narrative)
ax.set_title('Consumer Narrative Availability', fontsize=12, fontweight='bold')

# 3. Submission channels
ax = axes[1, 0]
submission_counts = mortgage_data['Submitted via'].value_counts()
ax.bar(range(len(submission_counts)), submission_counts.values, color='lightgreen')
ax.set_xticks(range(len(submission_counts)))
ax.set_xticklabels(submission_counts.index, rotation=45, ha='right')
ax.set_ylabel('Number of Complaints', fontsize=10)
ax.set_title('Complaints by Submission Channel', fontsize=12, fontweight='bold')
for i, v in enumerate(submission_counts.values):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontsize=9)

# 4. Timely response rate
ax = axes[1, 1]
timely_counts = mortgage_data['Timely response?'].value_counts()
ax.pie(timely_counts.values, labels=timely_counts.index, autopct='%1.1f%%', startangle=90, colors=['#99ff99', '#ffcc99'])
ax.set_title('Timely Response Rate', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n✓ Consumer narratives available: {mortgage_data['Consumer complaint narrative'].notna().sum():,} ({(mortgage_data['Consumer complaint narrative'].notna().sum()/len(mortgage_data)*100):.1f}%)")

In [ ]:
# Exploratory Data Analysis - Top Issues
print("TOP ISSUES IN MORTGAGE COMPLAINTS")
print("=" * 80)

issue_counts = mortgage_data['Issue'].value_counts()
print(f"\nTotal unique issues: {len(issue_counts)}")
print(f"\nTop 15 Issues:")
print(issue_counts.head(15))

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart - Top 10 issues
top_issues = issue_counts.head(10)
ax1.barh(range(len(top_issues)), top_issues.values, color='steelblue')
ax1.set_yticks(range(len(top_issues)))
ax1.set_yticklabels([label[:50] for label in top_issues.index], fontsize=9)
ax1.set_xlabel('Number of Complaints', fontsize=11)
ax1.set_title('Top 10 Mortgage Issues', fontsize=13, fontweight='bold')
ax1.invert_yaxis()
for i, v in enumerate(top_issues.values):
    ax1.text(v + 200, i, f'{v:,}', va='center', fontsize=9)

# Pie chart - Top 5 issues vs Others
top5 = issue_counts.head(5)
other_count = issue_counts[5:].sum()
pie_data = list(top5.values) + [other_count]
pie_labels = [label[:30] + '...' if len(label) > 30 else label for label in top5.index] + ['Others']
colors = sns.color_palette('Set2', len(pie_data))

ax2.pie(pie_data, labels=pie_labels, autopct='%1.1f%%', startangle=90, colors=colors)
ax2.set_title('Issue Distribution (Top 5 vs Others)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Data Quality Assessment
print("DATA QUALITY REPORT")
print("=" * 80)

# 1. Missing values
print("\n1. MISSING VALUES ANALYSIS")
print("-" * 80)
missing = mortgage_data.isnull().sum()
missing_pct = (missing / len(mortgage_data)) * 100
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing_Count': missing.values,
    'Missing_Percentage': missing_pct.values
}).sort_values('Missing_Count', ascending=False)

print(missing_df[missing_df['Missing_Count'] > 0].to_string(index=False))

# 2. Duplicates
print(f"\n2. DUPLICATE RECORDS")
print("-" * 80)
duplicates = mortgage_data.duplicated().sum()
print(f"Total duplicate rows: {duplicates:,}")

duplicate_ids = mortgage_data['Complaint ID'].duplicated().sum()
print(f"Duplicate Complaint IDs: {duplicate_ids:,}")

# 3. Data ranges
print(f"\n3. DATA RANGES")
print("-" * 80)
print(f"Date range: {mortgage_data['Date received'].min()} to {mortgage_data['Date received'].max()}")
print(f"Complaint IDs range: {mortgage_data['Complaint ID'].min():,} to {mortgage_data['Complaint ID'].max():,}")

In [ ]:
# Inspect column names and data types
print("Column Information:")
print("=" * 80)
print(f"\nColumns ({len(mortgage_data.columns)}):")
for i, col in enumerate(mortgage_data.columns, 1):
    dtype = mortgage_data[col].dtype
    null_count = mortgage_data[col].isnull().sum()
    null_pct = (null_count / len(mortgage_data)) * 100
    print(f"{i:2d}. {col:45s} | {str(dtype):10s} | Nulls: {null_count:6,} ({null_pct:5.1f}%)")

print("\n" + "=" * 80)
print("\nData Types Summary:")
mortgage_data.dtypes.value_counts()